<a href="https://colab.research.google.com/github/NataMaru/ML_for_people_tasks/blob/main/HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score


In [2]:
df=pd.read_csv('customer_segmentation_train.csv',index_col=0)
df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
ID,,,,,,,,,,
462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [3]:
#кодуємо букви таргета в числа
mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['Segmentation']= df['Segmentation'].map(mapping)


In [4]:
# % пропущенних значень
missing_data = df.isna().mean() * 100
print(missing_data)

Gender              0.000000
Ever_Married        1.735250
Age                 0.000000
Graduated           0.966782
Profession          1.536936
Work_Experience    10.275161
Spending_Score      0.000000
Family_Size         4.152206
Var_1               0.941993
Segmentation        0.000000
dtype: float64


In [5]:
#Gender заміна на 0 та 1
#Ever_Married, Graduated, No - 0 , Yes=1б пропуски модою замінити
#Profession ,Var_1- пропуск замінити unknown,  onehot encoder
#Work_Experience. Family_Size - median заміна пропусків


In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42,stratify=df['Segmentation'])
# Створюємо трен. і вал. набори
input_cols=list(df.columns)[:-1]
target_col='Segmentation'
train_inputs=train_df[input_cols].copy()
train_targets=train_df[target_col].copy()
test_inputs=test_df[input_cols].copy()
test_targets=test_df[target_col].copy()

In [7]:
train_inputs.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
ID,,,,,,,,,
465905,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6
462903,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6
467901,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6
463613,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6
459859,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7


In [8]:
# пайплайн для кодування Spending_Score
spending_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Average', 'High']]))
])

# Пайплайн для бінарних категорій (Gender, Ever_Married, Graduated)

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[
        ['Male', 'Female'], # Gender: Male=0, Female=1
        ['No', 'Yes'],      # Ever_Married: No=0, Yes=1
        ['No', 'Yes']       # Graduated: No=0, Yes=1
    ]))
])

#Пайплайн для складних категорій (Profession, Var_1)

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Пайплайн для числових даних (Work_Experience, Family_Size)
# Заміна медіаною
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Збираємо все в ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('spending', spending_pipe, ['Spending_Score']),
    ('binary', binary_pipeline, ['Gender', 'Ever_Married', 'Graduated']),
    ('cat', categorical_pipeline, ['Profession', 'Var_1']),
    ('num', numeric_pipeline, ['Work_Experience', 'Family_Size'])
], remainder='passthrough')

# Додаємо препроцесор у фінальний пайплайн
transform_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()) # Масштабування після всіх трансформацій
])

In [9]:
transform_pipeline.fit(train_inputs, train_targets)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('spending',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(categories=[['Low',
                                                                                               'Average',
                                                                                               'High']]))]),
                                                  ['Spending_Score']),
                                                 ('binary',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   Ordin...
                                                  ['Gender', 'Ever_Married',
                                                   'Graduated']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='unknown',
                                                                                 strategy='constant')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Profession', 'Var_1']),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Work_Experience',
                                                   'Family_Size'])])),
                ('scaler', StandardScaler())])

In [10]:
train_transformed_inputs=transform_pipeline.transform(train_inputs)
test_transformed_inputs=transform_pipeline.transform(test_inputs)

In [11]:
new_column_names = transform_pipeline.named_steps['preprocessor'].get_feature_names_out()
new_column_names

array(['spending__Spending_Score', 'binary__Gender',
       'binary__Ever_Married', 'binary__Graduated',
       'cat__Profession_Artist', 'cat__Profession_Doctor',
       'cat__Profession_Engineer', 'cat__Profession_Entertainment',
       'cat__Profession_Executive', 'cat__Profession_Healthcare',
       'cat__Profession_Homemaker', 'cat__Profession_Lawyer',
       'cat__Profession_Marketing', 'cat__Profession_unknown',
       'cat__Var_1_Cat_1', 'cat__Var_1_Cat_2', 'cat__Var_1_Cat_3',
       'cat__Var_1_Cat_4', 'cat__Var_1_Cat_5', 'cat__Var_1_Cat_6',
       'cat__Var_1_Cat_7', 'cat__Var_1_unknown', 'num__Work_Experience',
       'num__Family_Size', 'remainder__Age'], dtype=object)

In [24]:
train_final_inputs = pd.DataFrame(train_transformed_inputs, columns=new_column_names)
test_final_inputs = pd.DataFrame(test_transformed_inputs, columns=new_column_names)
test_final_inputs.head()

,spending__Spending_Score,binary__Gender,binary__Ever_Married,binary__Graduated,cat__Profession_Artist,cat__Profession_Doctor,cat__Profession_Engineer,cat__Profession_Entertainment,cat__Profession_Executive,cat__Profession_Healthcare,...,cat__Var_1_Cat_2,cat__Var_1_Cat_3,cat__Var_1_Cat_4,cat__Var_1_Cat_5,cat__Var_1_Cat_6,cat__Var_1_Cat_7,cat__Var_1_unknown,num__Work_Experience,num__Family_Size,remainder__Age
0,0.607787,-0.920704,0.823738,-1.304868,-0.671314,-0.310345,-0.308842,2.768228,-0.286318,-0.444300,...,-0.236549,-0.339781,2.564199,-0.097682,-1.364582,-0.15944,-0.09687,2.881103,0.102490,-0.155477
1,-0.737431,-0.920704,-1.213979,-1.304868,-0.671314,-0.310345,-0.308842,-0.361242,-0.286318,2.250731,...,-0.236549,-0.339781,-0.389985,-0.097682,0.732825,-0.15944,-0.09687,0.453842,0.769294,-1.355128
2,-0.737431,1.086126,0.823738,0.766361,-0.671314,3.222222,-0.308842,-0.361242,-0.286318,-0.444300,...,-0.236549,-0.339781,-0.389985,-0.097682,0.732825,-0.15944,-0.09687,-0.759788,-1.231118,-0.455390
3,-0.737431,1.086126,-1.213979,-1.304868,-0.671314,-0.310345,3.237898,-0.361242,-0.286318,-0.444300,...,-0.236549,-0.339781,-0.389985,-0.097682,0.732825,-0.15944,-0.09687,-0.759788,-0.564314,-0.995233
4,-0.737431,1.086126,-1.213979,-1.304868,-0.671314,-0.310345,-0.308842,-0.361242,-0.286318,2.250731,...,-0.236549,-0.339781,-0.389985,-0.097682,0.732825,-0.15944,-0.09687,-0.759788,2.102901,-1.535076


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [13]:
#Індекси категорій
categorical_indices = [i for i, name in enumerate(new_column_names)
                       if 'cat__' in name or 'binary__' in name or 'spending__' in name]


In [14]:
smotenc = SMOTENC(categorical_features=categorical_indices, random_state=42)
smotetomek = SMOTETomek(smote=smotenc, random_state=42)



In [15]:
train_smote_inputs, train_smote_targets = smotenc.fit_resample(train_final_inputs, train_targets)
test_smote_inputs, test_smote_targets = smotenc.fit_resample(test_final_inputs, test_targets)

In [16]:
train_smtomek_inputs, train_smtomek_targets = smotetomek.fit_resample(train_final_inputs, train_targets)
test_smtomek_inputs, test_smtomek_targets = smotetomek.fit_resample(test_final_inputs, test_targets)

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [17]:
model = OneVsRestClassifier(LogisticRegression())
model.fit(train_final_inputs, train_targets)
train_transform_predict=model.predict(train_final_inputs)
test_transform_predict=model.predict(test_final_inputs)


In [18]:
model_smote = OneVsRestClassifier(LogisticRegression())
model_smote.fit(train_smote_inputs, train_smote_targets)
train_smote_predict=model_smote.predict(train_smote_inputs)
test_smote_predict=model_smote.predict(test_smote_inputs)

In [19]:
model_smtomek = OneVsRestClassifier(LogisticRegression())
model_smtomek.fit(train_smtomek_inputs, train_smtomek_targets)
train_smtomek_predict=model_smtomek.predict(train_smtomek_inputs)
test_smtomek_predict=model_smtomek.predict(test_smtomek_inputs)

In [20]:
print('---------------------------------------------------------')
print ('                       TRAIN                            ')
print('---------------------------------------------------------')
print ('                      Basic                             ')
print('---------------------------------------------------------')
print(classification_report(train_transform_predict, train_targets))
print('---------------------------------------------------------')
print ('                       SMOTE'                            )
print('---------------------------------------------------------')
print(classification_report(train_smote_predict, train_smote_targets))
print('---------------------------------------------------------')
print ('                      Smote-Tomek                       ')
print('---------------------------------------------------------')
print(classification_report(train_smtomek_predict, train_smtomek_targets))

---------------------------------------------------------
                       TRAIN                            
---------------------------------------------------------
                      Basic                             
---------------------------------------------------------
              precision    recall  f1-score   support

           0       0.47      0.43      0.45      1759
           1       0.14      0.40      0.20       497
           2       0.65      0.48      0.55      2160
           3       0.71      0.63      0.67      2038

    accuracy                           0.51      6454
   macro avg       0.49      0.48      0.47      6454
weighted avg       0.58      0.51      0.53      6454

---------------------------------------------------------
                       SMOTE
---------------------------------------------------------
              precision    recall  f1-score   support

           0       0.48      0.43      0.46      2038
           1       0.20

In [21]:
print('---------------------------------------------------------')
print ('                       TEST                             ')
print('---------------------------------------------------------')
print ('                      Basic                             ')
print('---------------------------------------------------------')
print(classification_report(test_transform_predict, test_targets))
print('---------------------------------------------------------')
print ('                       SMOTE'                            )
print('---------------------------------------------------------')
print(classification_report(test_smote_predict, test_smote_targets))
print('---------------------------------------------------------')
print ('                      Smote-Tomek                       ')
print('---------------------------------------------------------')
print(classification_report(test_smtomek_predict, test_smtomek_targets))

---------------------------------------------------------
                       TEST                             
---------------------------------------------------------
                      Basic                             
---------------------------------------------------------
              precision    recall  f1-score   support

           0       0.45      0.42      0.44       424
           1       0.15      0.41      0.22       135
           2       0.64      0.48      0.55       520
           3       0.75      0.64      0.69       535

    accuracy                           0.51      1614
   macro avg       0.50      0.49      0.47      1614
weighted avg       0.59      0.51      0.54      1614

---------------------------------------------------------
                       SMOTE
---------------------------------------------------------
              precision    recall  f1-score   support

           0       0.47      0.44      0.45       483
           1       0.23

###висновки
1. Модель показала себе краще на збалансованих Smote-Tomek даних. Показник F1 є вищим. така сама ситуація і з precision та recall.
2. Також помітно, що клас 1 є проблемним, оскільки найвищий показник F1 на ньому 0.32, precision = 0.24, recall=0.50,а це дуже низькі показники. Скоріш за все це пов'язано з малою кіл-ті представників цього класу в даних, або він може бути подібним до якогось з інших наявних в наборі даних ( нижче приведений розподіл по кіл-ті представників кожного класу)

In [25]:
df['Segmentation'].value_counts().sort_values()

,count
Segmentation,
1,1858
2,1970
0,1972
3,2268
